# Cell-Cell Communication Analysis

This notebook demonstrates spatioloji_s's CCC module:

1. **Edge scoring** — score ligand-receptor communication on spatial graphs
2. **Significance testing** — analytical z-scores or subsampled permutation
3. **Interface-aware CCC** — stratify communication by spatial zone and measure gradients across tissue boundaries
4. **Morphology stratification** — compare communication across cell shape groups

No multi-layer pipeline — one simple scoring pass, then optional downstream analyses.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import spatioloji_s as sj
from spatioloji_s.ccc import CCCConfig, CCCResult, run_ccc

# Load your spatioloji object
# sp = sj.read_cosmx(...)  or  sj.read_xenium(...)

# Check available cell types and genes
print(sp.cell_meta["cell_type"].value_counts())
print(f"\n{len(sp.gene_index)} genes, {len(sp.cell_index)} cells")

---
## 1. Quick Start — One-line CCC

The simplest way to run CCC. Uses built-in LR database, default parameters, analytical significance testing.

In [ ]:
# Run with defaults — built-in LR pairs, analytical significance
result = run_ccc(sp)

# Top significant interactions
sig = result.scores[result.scores["fdr"] < 0.05].sort_values("mean_score", ascending=False)
print(f"{len(sig)} significant interactions (FDR < 0.05)")
sig.head(20)

---
## 2. Custom Configuration

### 2a. Using CellChatDB

Load a larger LR database from CellChatDB CSV.

In [ ]:
# Use CellChatDB — download from https://github.com/sqjin/CellChat
config = CCCConfig(
    db_source="cellchatdb",
    db_csv_path="path/to/CellChatDB_human.csv",  # adjust path
    group_col="cell_type",
    layer="log_normalized",       # use normalized expression
    min_pct=0.05,                 # LR genes must be expressed in >=5% of cells
)

result = run_ccc(sp, config)
print(f"{len(result.lr_pairs)} LR pairs tested")
print(f"{len(result.scores)} type-pair interactions scored")

### 2b. Configuring spatial parameters

The key spatial parameter is the communication radius for secreted/ECM signaling. Default is 200 um (~1000 pixels for Xenium), matching the paracrine diffusion range used by COMMOT.

In [ ]:
config = CCCConfig(
    group_col="cell_type",
    layer="log_normalized",

    # Spatial parameters
    secreted_radius=200.0,        # um — paracrine diffusion range
    ecm_radius=200.0,             # um — ECM signaling range
    buffer_distance=None,         # None = direct polygon contact for juxtacrine

    # Distance decay (auto-estimated if None)
    sigma_secreted=None,          # exponential decay parameter
    sigma_ecm=None,

    # Filter to specific signaling types
    lr_types=["secreted", "ecm"],  # or ["juxtacrine"] or None for all
)

result = run_ccc(sp, config)

### 2c. Significance testing

Two methods available:
- **Analytical** (default) — z-score from null distribution moments. Scales to 1M+ cells in seconds.
- **Permutation** — subsampled cells, shuffled labels. More robust for rare cell types.

In [ ]:
# Permutation test (more robust, slower)
config_perm = CCCConfig(
    group_col="cell_type",
    layer="log_normalized",
    test_method="permutation",
    n_subsample=10000,            # subsample cells for speed
    n_permutations=1000,          # number of label shuffles
    seed=42,
)

result_perm = run_ccc(sp, config_perm)

# Compare analytical vs permutation p-values
comparison = result.scores[["lr_name", "sender_type", "receiver_type", "pvalue"]].rename(
    columns={"pvalue": "pvalue_analytical"}
).merge(
    result_perm.scores[["lr_name", "sender_type", "receiver_type", "pvalue"]].rename(
        columns={"pvalue": "pvalue_permutation"}
    ),
    on=["lr_name", "sender_type", "receiver_type"],
)
print(comparison.head(10))

### 2d. Filter to specific cell types

Focus on interactions between specific sender and receiver types.

In [ ]:
# Only score Tumor -> immune interactions
config_filtered = CCCConfig(
    group_col="cell_type",
    layer="log_normalized",
    sender_types=["Tumor"],
    receiver_types=["CD8_T", "Macrophage", "B_cell"],
)

result_filtered = run_ccc(sp, config_filtered)
print(result_filtered.scores.sort_values("mean_score", ascending=False).head(10))

---
## 3. Exploring Results

### 3a. Score summary table

In [ ]:
# Significant interactions sorted by score
sig = result.scores[result.scores["fdr"] < 0.05].sort_values("mean_score", ascending=False)
print(f"{len(sig)} significant interactions\n")
print(sig[["lr_name", "sender_type", "receiver_type", "mean_score", "n_edges", "pvalue", "fdr"]].to_string())

### 3b. Cell-level scores

Per-cell sender and receiver scores for each LR pair. Useful for spatial visualization.

In [ ]:
# Cell-level scores
print(f"Cell scores shape: {result.cell_scores.shape}")
print(f"Columns: {list(result.cell_scores.columns)}")

# Pick a top LR pair and visualize sender/receiver scores spatially
top_lr = sig.iloc[0]["lr_name"] if len(sig) > 0 else result.scores.iloc[0]["lr_name"]
sender_col = f"{top_lr}_sender"
receiver_col = f"{top_lr}_receiver"

if sender_col in result.cell_scores.columns:
    sp._cell_meta[f"ccc_{sender_col}"] = result.cell_scores[sender_col].reindex(sp.cell_index).fillna(0).values
    sp._cell_meta[f"ccc_{receiver_col}"] = result.cell_scores[receiver_col].reindex(sp.cell_index).fillna(0).values

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    sj.visualization.xenium_plot_spatial(
        sp, f"ccc_{sender_col}", ax=axes[0], show=False,
        cmap="Reds", title=f"{top_lr} — Sender score",
    )
    sj.visualization.xenium_plot_spatial(
        sp, f"ccc_{receiver_col}", ax=axes[1], show=False,
        cmap="Blues", title=f"{top_lr} — Receiver score",
    )
    plt.tight_layout()
    plt.show()

---
## 4. Interface-Aware CCC

**What makes spatioloji_s unique:** stratify communication by spatial zone relative to a tissue boundary. No other CCC tool does this.

Requires an `InterfaceResult` from `identify_interface()`.

### 4a. Zone comparison

Compare communication strength at the interface vs. tumor interior vs. stroma interior.

In [ ]:
import spatioloji_s.spatial.point as spoint

# Step 1: Identify interface
iface = spoint.identify_interface(
    sp, group_col="cell_type",
    region_a="Tumor", region_b="Stroma",
    grid_resolution=50,
)

# Step 2: Run CCC with interface
config_iface = CCCConfig(
    group_col="cell_type",
    layer="log_normalized",
    interface_result=iface,
    n_distance_bins=20,
)

result_iface = run_ccc(sp, config_iface)

In [ ]:
# Zone comparison: which interactions are enriched at the interface?
zones = result_iface.zone_comparison
print("Zone comparison:")
print(zones[["lr_name", "sender_type", "receiver_type", "zone", "mean_score", "fold_change"]].to_string())

# Interactions strongest at the interface
interface_enriched = zones[zones["zone"] == "interface"].sort_values("fold_change", ascending=False)
print(f"\nTop interactions enriched at interface (fold_change > 1 = stronger at boundary):")
print(interface_enriched.head(10))

### 4b. Communication gradient

How does communication strength change as you move away from the interface?

- Positive slope → communication increases toward tumor side
- Negative slope → communication increases toward stroma side
- Flat → no spatial trend

In [ ]:
# Communication gradient across interface
grad = result_iface.zone_gradient
print("Communication gradient (score ~ distance to interface):")
print(grad[["lr_name", "sender_type", "receiver_type", "slope", "pvalue", "r2", "trend"]].to_string())

# Interactions with significant spatial gradients
sig_grad = grad[grad["pvalue"] < 0.05].sort_values("slope", key=abs, ascending=False)
print(f"\n{len(sig_grad)} interactions with significant gradient (p < 0.05)")
print(sig_grad.head(10))

---
## 5. Morphology Stratification

**Another spatioloji_s-unique feature:** compare communication patterns across cell shape groups.

Requires a categorical morphology column in `cell_meta` (e.g. from `classify_morphology()` or manual annotation).

In [ ]:
# First compute morphology if not already done
# sj.spatial.polygon.compute_morphology(sp)
# sj.spatial.polygon.classify_morphology(sp)

# Run CCC with morphology stratification
config_morph = CCCConfig(
    group_col="cell_type",
    layer="log_normalized",
    morphology_col="morph_class",   # or "morphology_class" from classify_morphology
)

result_morph = run_ccc(sp, config_morph)

# Compare: do elongated cells communicate differently than round cells?
morph = result_morph.morphology_comparison
print("Morphology comparison:")
print(morph[["lr_name", "sender_type", "receiver_type", "morphology_group", "mean_score", "fold_change"]].to_string())

# Interactions where morphology matters most
morph_diff = morph.groupby(["lr_name", "sender_type", "receiver_type"])["fold_change"].agg(lambda x: x.max() - x.min())
top_morph = morph_diff.sort_values(ascending=False).head(10)
print(f"\nTop interactions with largest morphology effect:")
print(top_morph)

---
## 6. Full Pipeline — Interface + Morphology Together

In [ ]:
# Everything together in one run
config_full = CCCConfig(
    group_col="cell_type",
    layer="log_normalized",

    # Spatial
    secreted_radius=200.0,
    ecm_radius=200.0,

    # Significance
    test_method="analytical",

    # Interface-aware
    interface_result=iface,
    n_distance_bins=20,

    # Morphology
    morphology_col="morph_class",
)

result_full = run_ccc(sp, config_full)

print(f"Runtime: {result_full.runtime_seconds:.1f}s")
print(f"Cells: {result_full.n_cells:,}")
print(f"Edges: {result_full.n_edges:,}")
print(f"LR pairs: {len(result_full.lr_pairs)}")
print(f"Interactions scored: {len(result_full.scores)}")
print(f"Significant (FDR<0.05): {(result_full.scores['fdr'] < 0.05).sum()}")
print(f"Zone comparison: {len(result_full.zone_comparison)} rows")
print(f"Gradient results: {len(result_full.zone_gradient)} rows")
print(f"Morphology comparison: {len(result_full.morphology_comparison)} rows")

---
## 7. Using the Low-Level API

For custom analyses, use the scoring functions directly instead of `run_ccc`.

In [ ]:
from spatioloji_s.ccc.database import LRPair, load_lr_database, filter_to_expressed
from spatioloji_s.ccc.scoring import score_edges, aggregate_scores, test_significance
from spatioloji_s.ccc.zones import compare_zones, communication_gradient, compare_morphology
from spatioloji_s.spatial.point.graph import build_radius_graph

# 1. Load and filter LR pairs
pairs = load_lr_database("builtin")
pairs = filter_to_expressed(pairs, sp, min_pct=0.05)
print(f"{len(pairs)} expressed LR pairs")

# 2. Build graph
graph = build_radius_graph(sp, radius=200, coord_type="global")

# 3. Score edges
edges = score_edges(sp, pairs, graph_diffusible=graph, group_col="cell_type", layer="log_normalized")
print(f"{len(edges)} edges scored")

# 4. Aggregate
summary, cell_scores = aggregate_scores(edges, sp, group_col="cell_type")

# 5. Test significance
summary = test_significance(summary, edges, sp, group_col="cell_type", method="analytical")

# 6. Zone comparison (requires interface result)
zone_df = compare_zones(edges, sp, iface, group_col="cell_type")

# 7. Communication gradient
grad_df = communication_gradient(edges, sp, iface, group_col="cell_type", n_bins=20)

# 8. Morphology comparison
morph_df = compare_morphology(edges, sp, morphology_col="morph_class", group_col="cell_type")

---
## Quick Reference

```python
from spatioloji_s.ccc import CCCConfig, run_ccc

# -- Basic run (built-in LR pairs, analytical significance) --
result = run_ccc(sp)

# -- CellChatDB --
result = run_ccc(sp, CCCConfig(db_source="cellchatdb", db_csv_path="CellChatDB.csv"))

# -- Custom spatial radius --
result = run_ccc(sp, CCCConfig(secreted_radius=300.0, ecm_radius=300.0))

# -- Permutation significance --
result = run_ccc(sp, CCCConfig(test_method="permutation", n_subsample=10000))

# -- Filter cell types --
result = run_ccc(sp, CCCConfig(sender_types=["Tumor"], receiver_types=["CD8_T"]))

# -- Interface-aware --
iface = spoint.identify_interface(sp, group_col="cell_type", region_a="Tumor", region_b="Stroma")
result = run_ccc(sp, CCCConfig(interface_result=iface))
# result.zone_comparison  → scores per zone (interface / interior_a / interior_b)
# result.zone_gradient    → communication strength vs distance to interface

# -- Morphology stratification --
result = run_ccc(sp, CCCConfig(morphology_col="morph_class"))
# result.morphology_comparison  → scores per morphology group

# -- Custom LR pairs (bypass database) --
from spatioloji_s.ccc.database import LRPair
my_pairs = [LRPair("MYL_MYR", "MY_LIGAND", "MY_RECEPTOR", "custom", "secreted")]
result = run_ccc(sp, CCCConfig(), lr_pairs=my_pairs)
```

## CCCResult fields

| Field | Type | Description |
|-------|------|-------------|
| `scores` | DataFrame | LR x sender_type x receiver_type → mean_score, sum_score, n_edges, pvalue, fdr |
| `cell_scores` | DataFrame | Per-cell sender/receiver scores for each LR pair |
| `lr_pairs` | list[LRPair] | LR pairs used |
| `zone_comparison` | DataFrame or None | Scores per spatial zone + fold_change |
| `zone_gradient` | DataFrame or None | Communication strength gradient: slope, pvalue, trend |
| `morphology_comparison` | DataFrame or None | Scores per morphology group + fold_change |
| `config` | CCCConfig | Configuration used |
| `n_cells` | int | Number of cells |
| `n_edges` | int | Number of edges scored |
| `runtime_seconds` | float | Total runtime |

## Scoring formula

```
score(i → j, LR) = sqrt(L_i × R_j) × w_ij
```

| Signaling type | Weight `w_ij` | Graph |
|---------------|--------------|-------|
| Juxtacrine | `contact_fraction(i→j) × contact_fraction(j→i)` | Polygon buffer graph |
| Secreted | `exp(-distance / sigma)` | Radius graph (200 um) |
| ECM | `exp(-distance / sigma)` | Radius graph (200 um) |